# TFCE tutorial

The core idea of TFCE is that you don't threshold the statistical map at any single cluster-forming threshold. Instead, for each voxel, you integrate a score across all possible thresholds, weighting by both the height of the signal and the spatial extent (cluster size) of the supra-threshold region at each threshold level. The formula accumulates contributions of the form $e(h)^E \cdot h^H$ across thresholds $h$, where $e(h)$ is the extent of the cluster containing that voxel at threshold $h$, and $E$ and $H$ are fixed parameters (typically $E = 0.5$, $H = 2$).

So the TFCE procedure doesn't operate on the raw t-value at a single voxel in isolation. It produces a transformed statistic for each voxel that reflects both its height and its spatial context (how much it's part of clusters at various thresholds). This is what makes it "cluster enhanced" without requiring an arbitrary cluster-forming threshold.

## How the p-value is computed

For each of the 64 sign-flip permutations, you:  
 1. Flip signs on the selected subjects' contrast maps  
     - For a given permutation, each subject is either flipped or not flipped as a whole. This makes sense from the logic of the permutation test: under the null hypothesis that the group mean effect is zero, each subject's contrast map is equally likely to be positive or negative. Flipping a subject's sign simulates what the data could have looked like if that subject's effect had gone the other direction. The spatial structure within each subject's map is preserved, which is important for the TFCE transformation since it relies on the spatial extent of clusters.    
 2. Compute the group t-map  
 3. Apply the TFCE transformation to the entire t-map  
 4. Record the maximum TFCE score across all voxels (this is what controls the family-wise error rate)  

Then for a given voxel, the corrected p-value is the proportion of permutations whose maximum TFCE score (across the whole brain) equals or exceeds the observed TFCE score at that voxel. Using the max statistic across the brain for comparison is what provides family-wise error correction.

## Numerical example

## Setup: 3 subjects, 5 neighboring voxels

### Observed contrast maps (already in group space)

|         | v1 | v2 | v3 | v4 | v5 |
|---------|----|----|----|----|-----|
| Sub1    |  2 |  3 |  4 |  3 |  1  |
| Sub2    |  1 |  2 |  3 |  2 |  0  |
| Sub3    |  1 |  2 |  3 |  2 |  1  |

---

## Step 1: Compute the group t-map

For each voxel, run a one-sample t-test across the 3 subjects.

**Example for v3:** values = [4, 3, 3], mean = 3.33, SE = 0.33, t = 10.0

| v1   | v2   | v3   | v4   | v5   |
|------|------|------|------|------|
| 3.46 | 6.06 | 10.0 | 6.06 | 1.73 |

---

## Step 2: Apply the TFCE transformation

TFCE formula for each voxel:

$$\text{TFCE}(v) = \sum_{h} e(h)^E \cdot h^H \cdot dh$$

where:
- $h$ = threshold being swept from 0 to the max t-value
- $e(h)$ = spatial extent (number of voxels) of the cluster containing voxel $v$ at threshold $h$
- $E = 0.5$, $H = 2$ (standard parameters)
- $dh$ = threshold step size (using dh = 1 here for simplicity)

### Sweeping through thresholds

**h = 1:** All voxels exceed 1 → one cluster of extent 5

| v1 | v2 | v3 | v4 | v5 |
|----|----|----|----|----|
| ✓  | ✓  | ✓  | ✓  | ✓  |

Contribution: $\sqrt{5} \cdot 1^2 = 2.24$ for all voxels

---

**h = 2:** v5 drops out (1.73 < 2) → cluster of extent 4

| v1 | v2 | v3 | v4 | v5 |
|----|----|----|----|----|
| ✓  | ✓  | ✓  | ✓  |    |

Contribution: $\sqrt{4} \cdot 2^2 = 8.0$ for v1–v4

---

**h = 3:** Same cluster, extent 4 (v1 = 3.46 still survives)

| v1 | v2 | v3 | v4 | v5 |
|----|----|----|----|----|
| ✓  | ✓  | ✓  | ✓  |    |

Contribution: $\sqrt{4} \cdot 3^2 = 18.0$ for v1–v4

---

**h = 4:** v1 drops out (3.46 < 4) → cluster of extent 3

| v1 | v2 | v3 | v4 | v5 |
|----|----|----|----|----|
|    | ✓  | ✓  | ✓  |    |

Contribution: $\sqrt{3} \cdot 4^2 = 27.7$ for v2–v4

---

**h = 5:** Same cluster, extent 3

Contribution: $\sqrt{3} \cdot 5^2 = 43.3$ for v2–v4

---

**h = 6:** Same cluster, extent 3

Contribution: $\sqrt{3} \cdot 6^2 = 62.4$ for v2–v4

---

**h = 7:** v2 and v4 drop out (6.06 < 7) → only v3 survives, extent 1

| v1 | v2 | v3 | v4 | v5 |
|----|----|----|----|----|
|    |    | ✓  |    |    |

Contribution: $\sqrt{1} \cdot 7^2 = 49.0$ for v3

---

**h = 8, 9, 10:** Only v3 survives, extent 1

- h = 8: $1 \cdot 64 = 64.0$
- h = 9: $1 \cdot 81 = 81.0$
- h = 10: $1 \cdot 100 = 100.0$

---

### Summing contributions across thresholds

| Voxel | Thresholds active | Total TFCE score |
|-------|-------------------|------------------|
| v1    | h = 1, 2, 3       | 2.24 + 8.0 + 18.0 = **28.2** |
| v2    | h = 1, 2, 3, 4, 5, 6 | 2.24 + 8.0 + 18.0 + 27.7 + 43.3 + 62.4 = **161.6** |
| v3    | h = 1, 2, 3, 4, 5, 6, 7, 8, 9, 10 | 2.24 + 8.0 + 18.0 + 27.7 + 43.3 + 62.4 + 49.0 + 64.0 + 81.0 + 100.0 = **455.6** |
| v4    | h = 1, 2, 3, 4, 5, 6 | 2.24 + 8.0 + 18.0 + 27.7 + 43.3 + 62.4 = **161.6** |
| v5    | h = 1              | 2.24 = **2.2** |

**Notice:** v3 has the highest TFCE score because it is both tall (t = 10) and consistently part of a cluster. v2 and v4 benefit substantially from being neighbors of v3 (cluster extent boosts their scores). v5 barely accumulates anything.

---

## Step 3: Permutation inference

With 3 subjects, there are $2^3 = 8$ sign-flip permutations.

For each permutation:
1. Flip the signs of all voxels for the selected subjects
2. Recompute the group t-map
3. Apply the TFCE transformation to the full map
4. Record the **maximum TFCE score** across all voxels

### Example permutation: flip Sub2

|         | v1  | v2  | v3  | v4  | v5 |
|---------|-----|-----|-----|-----|----|
| Sub1    |  2  |  3  |  4  |  3  |  1 |
| Sub2    | **-1** | **-2** | **-3** | **-2** | **0** |
| Sub3    |  1  |  2  |  3  |  2  |  1 |

New means are much smaller → new t-map has lower values → new TFCE scores are lower → the max TFCE from this permutation is lower.

### Building the null distribution

After running all 8 permutations, you have 8 max-TFCE values forming the null distribution.

---

## Step 4: Corrected p-value

For any voxel (e.g., v3 with observed TFCE = 455.6):

$$p_{\text{corrected}}(v3) = \frac{\text{number of permutations where max-TFCE} \geq 455.6}{8}$$

Because the comparison is against the **maximum** TFCE across the whole map in each permutation, this provides family-wise error rate correction.